1.LOAD DATASET

In [1]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

PLATFORM = "twitter"
SEEDS = [42, 123, 2024, 7, 99]
DATA_DIR = "../data_preprocess/processed_data/ml_data/"

results = []

for seed in SEEDS:
    train_df = pd.read_csv(f"{DATA_DIR}{PLATFORM}_train_seed{seed}.csv")
    test_df  = pd.read_csv(f"{DATA_DIR}{PLATFORM}_test_seed{seed}.csv")

    neg, pos = train_df["popularity"].value_counts()[0], train_df["popularity"].value_counts()[1]
    ratio = neg / pos

    X_train = train_df.drop(columns=["post_id", "user_id", "popularity"])
    y_train = train_df["popularity"]
    X_test  = test_df.drop(columns=["post_id", "user_id", "popularity"])
    y_test  = test_df["popularity"]

    model = XGBClassifier(
        objective="binary:logistic", eval_metric="logloss",
        scale_pos_weight=ratio, n_estimators=500, learning_rate=0.02,
        max_depth=6, min_child_weight=1, gamma=0.1,
        subsample=0.8, colsample_bytree=0.5,
        random_state=seed, n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "seed": seed,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
    })
    print(f"[seed {seed}] done | F1={results[-1]['f1']:.4f} | ROC-AUC={results[-1]['roc_auc']:.4f}")

results_df = pd.DataFrame(results)
print("\n" + "="*60)
print(f"{PLATFORM.upper()} BASELINE — MEAN ± STD ACROSS {len(SEEDS)} SEEDS")
print("="*60)
print(results_df.set_index("seed"))
print("\nMean ± Std:")
summary = results_df.drop(columns="seed").agg(["mean", "std"])
print(summary)

[seed 42] done | F1=0.6224 | ROC-AUC=0.8930
[seed 123] done | F1=0.6152 | ROC-AUC=0.8862
[seed 2024] done | F1=0.6168 | ROC-AUC=0.8880
[seed 7] done | F1=0.6272 | ROC-AUC=0.8877
[seed 99] done | F1=0.6233 | ROC-AUC=0.8886

TWITTER BASELINE — MEAN ± STD ACROSS 5 SEEDS
      accuracy  precision    recall        f1   roc_auc
seed                                                   
42    0.800694   0.502912  0.816463  0.622430  0.892966
123   0.797449   0.497935  0.804783  0.615221  0.886185
2024  0.799351   0.500868  0.802558  0.616798  0.887960
7     0.807632   0.514042  0.804227  0.627196  0.887665
99    0.803827   0.507881  0.806452  0.623254  0.888559

Mean ± Std:
      accuracy  precision    recall        f1   roc_auc
mean  0.801791   0.504728  0.806897  0.620980  0.888667
std   0.004009   0.006345  0.005525  0.004913  0.002557


In [2]:
results_df.insert(0, "model", "baseline_metadata")
results_df.insert(0, "platform", PLATFORM)

import os
os.makedirs("../results", exist_ok=True)
results_df.to_csv(f"../results/{PLATFORM}_baseline_metadata.csv", index=False)
print(f"Saved: ../results/{PLATFORM}_baseline_metadata.csv")

Saved: ../results/twitter_baseline_metadata.csv
